# Lab 09: Deployment & Model Serving

## Overview

In this lab you will deploy a registered Unity Catalog model as a **Databricks Model Serving endpoint**, test it via the REST API, configure an **A/B traffic split** across two model versions, and run **batch inference** using the `ai_query()` SQL function.

| Topic | Detail |
|---|---|
| **Exam Domain** | Assembling and Deploying Apps (22%) |
| **Key Skills** | Serving endpoints · Scale-to-zero · A/B testing · `ai_query()` · Provisioned vs pay-per-token |

### What you will do
1. Deploy a Unity Catalog model to a Model Serving endpoint using the Databricks SDK.
2. Test the endpoint with a REST query.
3. Configure A/B testing with a 50/50 traffic split between two model versions.
4. Run batch inference from SQL using `ai_query()`.
5. Compare **Provisioned Throughput** vs **Pay-per-Token** (serverless) pricing models.

In [ ]:
%pip install databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG        = "genai_lab"          # Unity Catalog catalog name
SCHEMA         = "default"            # Schema inside the catalog
ENDPOINT_NAME  = "genai-lab-agent-endpoint"
MODEL_NAME     = f"{CATALOG}.{SCHEMA}.rag_agent"   # Fully-qualified UC model name
MODEL_VERSION  = 1

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"Endpoint : {ENDPOINT_NAME}")
print(f"Model    : {MODEL_NAME}  (version {MODEL_VERSION})")

## A. Deploy to Model Serving

The Databricks SDK `WorkspaceClient` wraps the Model Serving REST API.  
`create_and_wait` blocks until the endpoint reaches the `READY` state.  
Setting `scale_to_zero_enabled=True` ensures the endpoint scales down when idle, saving cost.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

served_entity = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=str(MODEL_VERSION),
    workload_size="Small",          # XSmall | Small | Medium | Large
    scale_to_zero_enabled=True,
)

try:
    endpoint = w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=[served_entity]),
    )
    print(f"Endpoint created: {endpoint.name}  state={endpoint.state.ready}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists — skipping creation.")
        endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
        print(f"Existing endpoint state: {endpoint.state.ready}")
    else:
        raise

## B. Test the Endpoint

Use `w.serving_endpoints.query` to send a single request using the `dataframe_records` format.  
The response contains `predictions` (or `choices` for Foundation Model API endpoints).

In [ ]:
import json

response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{"input": "What is the transformer architecture?"}],
)

# The SDK returns a QueryEndpointResponse object; convert to dict for display
response_dict = response.as_dict()
print(json.dumps(response_dict, indent=2))

## C. A/B Testing

Databricks Model Serving supports **traffic splitting** across multiple served entities within a single endpoint.  
Each `ServedEntityInput` receives a `traffic_percentage`; the values must sum to 100.

Use cases:
- Compare a fine-tuned model against the baseline without changing client code.
- Gradually roll out a new model version (canary release).
- Measure latency or quality differences under production traffic.

Traffic routing is handled transparently by the endpoint — callers always hit the same endpoint URL.

In [ ]:
from databricks.sdk.service.serving import TrafficConfig, Route

MODEL_VERSION_2 = 2   # Assumes version 2 has been registered in UC

served_v1 = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=str(MODEL_VERSION),
    workload_size="Small",
    scale_to_zero_enabled=True,
    name="rag_agent_v1",
)

served_v2 = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=str(MODEL_VERSION_2),
    workload_size="Small",
    scale_to_zero_enabled=True,
    name="rag_agent_v2",
)

traffic_config = TrafficConfig(
    routes=[
        Route(served_model_name="rag_agent_v1", traffic_percentage=50),
        Route(served_model_name="rag_agent_v2", traffic_percentage=50),
    ]
)

updated_endpoint = w.serving_endpoints.update_config_and_wait(
    name=ENDPOINT_NAME,
    served_entities=[served_v1, served_v2],
    traffic_config=traffic_config,
)

print(f"A/B config applied. Endpoint state: {updated_endpoint.state.ready}")
for route in updated_endpoint.config.traffic_config.routes:
    print(f"  {route.served_model_name}: {route.traffic_percentage}%")

## D. Batch Inference with `ai_query()`

`ai_query()` is a built-in Databricks SQL function that calls a Model Serving endpoint for every row in a query.  
It enables **batch inference directly in SQL** without writing Python loops or Spark UDFs.

**Syntax:**
```sql
ai_query(
    endpoint_name,   -- string literal: name of the serving endpoint
    request          -- column or expression passed as the model input
)
```

Ideal for scheduled batch jobs, Delta table enrichment, and offline evaluation pipelines.

In [ ]:
# Step 1: Create a small test table with questions
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.test_questions (
        id       INT,
        question STRING
    )
""")

spark.sql(f"""
    INSERT INTO {CATALOG}.{SCHEMA}.test_questions VALUES
        (1, 'What is the transformer architecture?'),
        (2, 'Explain attention mechanisms in neural networks.'),
        (3, 'What is retrieval-augmented generation?'),
        (4, 'How does LangChain differ from LlamaIndex?'),
        (5, 'What are the key components of a RAG pipeline?')
""")

# Step 2: Run batch inference using ai_query()
results_df = spark.sql(f"""
    SELECT
        id,
        question,
        ai_query(
            '{ENDPOINT_NAME}',
            question
        ) AS answer
    FROM {CATALOG}.{SCHEMA}.test_questions
    ORDER BY id
""")

display(results_df)

## E. Provisioned Throughput vs Pay-per-Token

Databricks offers two serving tiers for Foundation Models. Choosing the right one depends on your usage pattern.

### Pay-per-Token (Serverless)
- **Billed per token** processed (input + output).
- No upfront commitment; scales automatically to zero when idle.
- Best for: **development, testing, low/bursty traffic** workloads.
- Latency may vary under high concurrency.

### Provisioned Throughput
- **Billed per DBU/hour** regardless of actual token usage.
- Reserved compute: **guaranteed tokens per second (TPS)**, predictable latency.
- Best for: **production workloads with sustained, high-volume traffic**.
- Eliminates cold-start latency; SLA-backed availability.

### Cost Comparison

| Scenario | Pay-per-Token | Provisioned Throughput |
|---|---|---|
| 10 req/day, dev | Low cost | Wasteful (idle hours billed) |
| 1 000 req/min, prod | High variable cost | Lower per-token effective cost |
| Burst traffic | Handled automatically | May exceed provisioned TPS |
| Latency SLA | Not guaranteed | Guaranteed |

### When to Choose
| Choose | When |
|---|---|
| **Pay-per-Token** | Prototyping, internal tools, unpredictable demand |
| **Provisioned Throughput** | Customer-facing APIs, real-time dashboards, batch pipelines with SLA |

> **Exam tip:** The exam tests whether you can identify that `scale_to_zero_enabled` applies to **custom model serving**, not Foundation Model API endpoints, which are always serverless.

## Key Takeaways

| Concept | Summary |
|---|---|
| **Serving Endpoints** | Deploy UC-registered models via SDK or UI; endpoint URL is stable regardless of model version |
| **Scale-to-Zero** | Enabled per `ServedEntityInput`; reduces cost for low-traffic custom model endpoints |
| **A/B Testing** | Use `TrafficConfig` with multiple `ServedEntityInput` entries summing to 100% traffic |
| **`ai_query()` Batch** | SQL-native batch inference; joins model output directly to table columns |
| **Provisioned vs Serverless** | Choose based on traffic volume, latency SLA, and cost predictability |

---

See **[workbook.md](workbook.md)** for the architecture diagram, exam questions, and cost breakdown.